In [ ]:
#@markdown Setup dependencies (Colab only)
%%capture
try:
    import papyrus_scripts  # noqa: F401
except ImportError:
    !pip uninstall papyrus-scripts -y
    !pip install --upgrade papyrus-scripts --no-cache-dir
    get_ipython().kernel.do_shutdown(True)
try:
    import pyfamsa  # noqa: F401
except ImportError:
    !pip install pyfamsa

# 🧬 Protein descriptors: using ProDEC via Papyrus-scripts

This notebook covers [ProDEC](https://github.com/OlivierBeq/ProDEC), the library `papyrus-scripts` uses under the hood to turn a protein sequence into physicochemical-scale-based descriptors (VHSE, Z-scales, FASGAI, and dozens more), and the transforms built on top of them (average, auto-cross covariance, physicochemical distance transform, Fourier transform).

1. [🧩 Fetching protein descriptors via ProDEC](#1) — the raw, per-residue descriptors, and why the number of features they produce depends on protein length.
2. [🔁 Fixed-length transforms](#2) — `AVG`/`ACC`/`PDT` summarize a sequence of any length into a constant number of features; `FFT`, surprisingly, does not.
3. [🧵 Aligning short sequences for raw descriptors](#3) — how to get a fixed-length, *positionally meaningful* feature table out of the raw per-residue values, using a multiple sequence alignment on a handful of short (~100 residue) proteins.
4. [🗺️ Recap: alignment requirements at a glance](#4) — which of the above need every sequence to be pre-aligned to be usable/comparable, and which don't.
5. [🤖 Modelling with protein descriptors: `pcm()`](#5) — one shared model across every target, driven by `prot_descriptors=`; compares the descriptor/transform types above with gradient boosting, fits the same combined molecular + protein features with a `SingleTaskNNRegressor`, then contrasts that with the other multi-task approach — `MultiTaskNNClassifier` pivoting targets into output columns instead.

<a id="1"></a>
## 1. 🧩 Fetching protein descriptors via ProDEC

`prodec.ProteinDescriptors()` is a registry of every amino-acid-scale descriptor ProDEC knows about (VHSE, the various Z-scales, FASGAI, Kidera, MS-WHIM, ST-scale, ProtFP, BLOSUM, Sneath...). Each one assigns a small numeric vector to every amino acid; a protein's **raw** descriptor is just those per-residue vectors concatenated — so its length is `n_scales × sequence_length` and therefore differs between proteins of different lengths.

`papyrus-scripts` exposes them two ways:
- **🧩 object-oriented**: `protein_set.protein_descriptors(desc)` on any `ProteinSet` (e.g. from `.proteins()`), where `desc` is a ProDEC `Descriptor`/`Transform`, or the string `'unirep'`/`'custom'`.
- **📚 functional**: `read_protein_descriptors(desc_type=desc, ids=...)`, mirroring the molecular descriptor readers used elsewhere.

Both fetch the `Sequence` column for the targets involved and hand it to ProDEC — there's no separate download step, unlike molecular descriptors.

In [ ]:
from papyrus_scripts import PapyrusDataset

protein_set = (PapyrusDataset(version='latest', plusplus=True)
               .keep_protein_class([{'l2': 'Ligand-gated ion channels'},
                                    {'l3': 'SLC superfamily of solute carriers'}])
               .keep_organism('Rattus norvegicus (rat)')
               .proteins())

proteins = protein_set.to_dataframe()
proteins.select(['target_id', 'Organism', 'Length']).sort('Length')

In [ ]:
import prodec

registry = prodec.ProteinDescriptors()
print(f'📖 {len(registry.available_descriptors)} available ProDEC descriptors, '
      f'e.g.: {registry.available_descriptors[:6]}')

vhse = registry.get_descriptor('VHSE')
vhse.definition[0]  # the 8 VHSE scales assigned to every amino acid

In [ ]:
vhse_data = protein_set.protein_descriptors(vhse)
print(f'🧬 {vhse_data.shape[0]} proteins × {vhse_data.shape[1] - 1} columns '
      f'(from the longest protein: {(vhse_data.shape[1] - 1) // 8} residues × 8 VHSE scales)')
vhse_data.head()

That column count comes from the **longest** protein in the set — every shorter one is padded with missing values rather than aligned, so the raw descriptor isn't really comparable feature-by-feature across proteins. Let's make that explicit:

In [ ]:
vhse_pd = vhse_data.to_pandas().set_index('target_id')

usage = (vhse_pd.notna().sum(axis=1) // 8).rename('residues used').to_frame()
usage['Length'] = proteins.to_pandas().set_index('target_id').loc[usage.index, 'Length']
usage.sort_values('Length')

Every row uses exactly as many residues as its own length (as expected — nothing is wrong with the *values*), but the resulting table has `NaN`s wherever a protein is shorter than the longest one, and each protein's column *n* corresponds to a different residue. A raw per-residue descriptor is only directly usable across multiple proteins when every sequence is the same length — ideally an *aligned* one (see [section 3](#3)).

<a id="2"></a>
## 2. 🔁 Fixed-length transforms

ProDEC wraps every descriptor with four transforms that summarize the per-residue values into features that no longer scale with sequence length:

| `TransformType` | Idea | Constant-length? |
|---|---|---|
| `AVG` | average the scale values over `domains` equally-sized chunks of the sequence | ✅ always |
| `ACC` | auto- and cross-covariance between scales at a given `lag` | ✅ always |
| `PDT` | physicochemical distance transform between residues `lag` apart | ✅ always |
| `FFT` | Fourier transform of each scale's per-residue signal | ❌ scales with sequence length |

`prodec.Transform(transform_type, descriptor)` builds one; it plugs into `protein_descriptors()`/`read_protein_descriptors()`/`pcm(..., prot_descriptors=...)` exactly like a bare `Descriptor`.

In [ ]:
from prodec import Transform

transforms = {
    'AVG': Transform('AVG', vhse),
    'ACC': Transform('ACC', vhse),
    'PDT': Transform('PDT', vhse),
    'FFT': Transform('FFT', vhse),
}

for name, transform in transforms.items():
    data = protein_set.protein_descriptors(transform).to_pandas()
    n_features = data.shape[1] - 1
    n_missing = int(data.isna().sum().sum())
    print(f'{name:>4}: {n_features:>5} feature(s) for every one of the {data.shape[0]} proteins, '
          f'{n_missing} missing value(s)')

`AVG`, `ACC` and `PDT` come back with zero missing values — every protein, whatever its length, gets the same number of features. `FFT` doesn't: its output length grows with the sequence (roughly `4 × length` per scale here), so shorter proteins are still padded with `NaN` once combined into one table, just like the raw descriptor was.

> 💡 `AVG`/`ACC`/`PDT` are exactly what's normally passed as `prot_descriptors=` to `pcm()` (see [advanced_querying.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/advanced_querying.ipynb)) when protein sequences aren't pre-aligned — no extra step needed.

<a id="3"></a>
## 3. 🧵 Aligning short sequences for raw descriptors

If you specifically want the **raw**, un-transformed per-residue values as fixed-length features (rather than an `AVG`/`ACC`/`PDT` summary) — e.g. to preserve full positional resolution for a set of homologous proteins — every sequence must not just be the *same length*, but *aligned*, so that column *i* is the same structural/evolutionary position in every protein. Padding shorter sequences with missing values (as ProDEC does by default, [section 1](#1)) does not achieve that.

To keep this fast and easy to follow, let's grab a handful of the shortest protein sequences in Papyrus (~100 residues or fewer) instead of the full-length transporters used above.

In [ ]:
import polars as pl

short_proteins = (PapyrusDataset(version='latest')
                  .keep_organism('Homo sapiens (human)')
                  .keep_protein_class({'l1': 'Enzyme'})
                  .proteins()
                  .agg()
                  .filter(pl.col('Length') <= 150)
                  .sort('Length', descending=True)
                  .head(10))
short_proteins.select(['target_id', 'Organism', 'Length'])

First, the naive approach — handing the unaligned sequences straight to `vhse.pandas_get()`:

In [ ]:
vhse_naive = vhse.pandas_get(short_proteins['Sequence'].to_list(),
                             short_proteins['target_id'].to_list(),
                             gaps=0)
n_features = vhse_naive.shape[1] - 1
print(f'🧬 {n_features} columns (from the longest of these {short_proteins["Length"].max()}-residue sequences)')
print(f'❌ {int(vhse_naive.isna().sum().sum())} missing values — and even the non-missing ones '
      f'aren\'t comparable position-by-position between proteins of different lengths')

Now align the sequences first, with [pyFAMSA](https://github.com/althonos/pyFAMSA) (Python bindings to [FAMSA](https://github.com/refresh-bio/FAMSA), a fast multiple sequence aligner) — no external binary to install, and well suited to a handful of short sequences like this one:

In [ ]:
from pyfamsa import Aligner, Sequence

records = [Sequence(target_id.encode(), sequence.encode())
          for target_id, sequence in short_proteins.select(['target_id', 'Sequence']).iter_rows()]

alignment = Aligner().align(records)

aligned_ids = [s.id.decode() for s in alignment]
aligned_seqs = [s.sequence.decode() for s in alignment]  # same length, padded with '-'
print(f'📏 alignment length: {len(aligned_seqs[0])} columns for all {len(aligned_seqs)} sequences')
for seq_id, seq in zip(aligned_ids, aligned_seqs, strict=True):
    print(f'{seq_id:>15}: {seq}')

Every sequence is now the same length, `'-'`-gapped where it doesn't reach that length — so the raw descriptor can be computed directly with ProDEC (bypassing `protein_descriptors()`, which would reject these sequences: it validates them against the *ungapped* alphabet). `gaps=0` fills every gap position with `0.0`, a neutral value on these standardized scales:

In [ ]:
vhse_aligned = vhse.pandas_get(aligned_seqs, aligned_ids, gaps=0)
n_features = vhse_aligned.shape[1] - 1
print(f'✅ {n_features} feature(s) for every one of the {vhse_aligned.shape[0]} proteins, '
      f'{int(vhse_aligned.isna().sum().sum())} missing value(s)')
vhse_aligned.head()

Column *i* now means the same aligned position for every protein in the set — a fixed-length, positionally meaningful feature table. Note that `pcm()`'s own `prot_descriptors=` only accepts a ProDEC `Descriptor`/`Transform` or `'unirep'` (it always fetches and computes the descriptor itself from the *ungapped* sequences) — so this hand-aligned table is for standalone use (e.g. a similarity/clustering analysis of this small protein family, or as a custom feature block you join into your own modelling pipeline), not a drop-in `pcm()` argument.

<a id="4"></a>
## 4. 🗺️ Recap: alignment requirements at a glance

Pulling together what sections 1–3 showed empirically:

| | Fixed length regardless of protein length? | Needs aligned input to be *comparable* across proteins? | Safe as `prot_descriptors=` for `pcm()`/`read_protein_descriptors()` with unaligned, varying-length sequences? |
|---|---|---|---|
| Raw `Descriptor` (VHSE, Z-scales, FASGAI, …) | ❌ no — `n_scales × length` | ✅ yes — column *i* is a different residue in every protein otherwise (section 1) | ⚠️ only if every sequence is already the same length (e.g. hand-aligned, section 3) |
| `Transform('AVG'/'ACC'/'PDT', desc)` | ✅ yes, always | ❌ no — built to summarize any-length sequences into the same feature set | ✅ yes — this is the normal choice |
| `Transform('FFT', desc)` | ❌ no — output length still scales with protein length | ✅ yes — same caveat as the raw descriptor | ❌ no |
| `'unirep'` | ✅ yes, always (pretrained embedding) | ❌ no | ✅ yes |

In short: reach for `AVG`/`ACC`/`PDT` (or `'unirep'`) whenever your proteins aren't pre-aligned or aren't the same length — which covers most PCM use cases across a diverse target family. Reach for the raw descriptor only when you've aligned the sequences yourself first, typically for a small, homologous set where positional resolution matters more than a fixed, general-purpose feature set.

<a id="5"></a>
## 5. 🤖 Modelling with protein descriptors: `pcm()`

`qsar()` and `pcm()` both merge molecular descriptors onto the activity data, split it, cross-validate a model and evaluate it — but only `pcm()` also merges a *protein* descriptor, which is why it's the one this notebook demonstrates:

- **`qsar()`** loops over every `target_id` present in the data and fits one *independent* model per target, using **molecular descriptors only**. There's no `prot_descriptors=` argument at all: within a single target's model the protein never varies, so a protein descriptor would just be a constant column — it has nothing to do with protein descriptors, so it isn't demonstrated here (see [advanced_querying.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/advanced_querying.ipynb) for it, along with splitting strategies, `yscramble` and more).
- **`pcm()`** fits **one shared model** across *every* target in the data at once, by additionally merging a protein descriptor (`prot_descriptors=`) onto each row. The model sees the protein descriptor as extra features, so it can learn how activity depends on target identity/similarity and, ideally, generalize to targets it wasn't trained on (given a suitable descriptor and enough related targets).

That's exactly where the choice of protein descriptor from sections 1–4 matters: `pcm()` needs one that's fixed-length across all targets in the data, so `AVG`/`ACC`/`PDT` `Transform`s (or `'unirep'`) are the natural fit, plugged in directly as `prot_descriptors=`.

We'll fit against every target in Papyrus's TK protein kinase group with high-quality Ki data — a larger, more heterogeneous target set than sections 1–3's transporters, exactly the kind of family a shared PCM model is meant to generalize across.

In [ ]:
import xgboost

model_dataset = (PapyrusDataset(version='latest', plusplus=False)
                  .keep_protein_class({'l4': 'TK protein kinase group'})
                  # .keep_accession(['P31645', 'P31652'])  # human & rat SLC6A4
                  .keep_quality('high')
                  .keep_activity_type(['Ki']))

model_data = model_dataset.to_dataframe()
model_data.shape

### 🌳 `pcm()` with gradient boosting

Every target is modelled *together*, one shared model, distinguished only by whatever `prot_descriptors=` encodes about each protein. Let's compare the fixed-length choices from [section 4](#4) — the very same `AVG`/`ACC`/`PDT` `Transform` objects built on VHSE in [section 2](#2), plus `'unirep'`:

In [ ]:
import pandas as pd

from papyrus_scripts.modelling import pcm

prot_descriptor_options = {
    'unirep': 'unirep',
    'VHSE-AVG': transforms['AVG'],  # built in section 2
    'VHSE-ACC': transforms['ACC'],
    'VHSE-PDT': transforms['PDT'],
}

pcm_summary = {}
for name, prot_desc in prot_descriptor_options.items():
    pcm_results, _ = pcm(data=model_data,
                          endpoint='pchembl_value_Mean',
                          mol_descriptors='mold2',
                          prot_descriptors=prot_desc,
                          model=xgboost.XGBRegressor(verbosity=0),
                          folds=5,
                          split_by='random',
                          random_state=1234,
                          verbose=False)
    pcm_summary[name] = pcm_results.loc['Test set'].copy()

pd.DataFrame(pcm_summary).T

### 🧠 The same combined features, fit with a neural net

`pcm()` only accepts scikit-learn-compatible estimators — the custom DNN classes in `papyrus_scripts.neuralnet` need `set_architecture()` (and, for early stopping, `set_validation()`) called explicitly before `.fit()`, which `pcm()`'s internal cross-validation loop doesn't do (see [advanced_querying.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/advanced_querying.ipynb) for the full explanation). To feed the exact same *molecular + protein* descriptor combination `pcm()` builds internally (merge molecular descriptors on the compound id, then merge a protein descriptor on `target_id`) into `SingleTaskNNRegressor` — a single continuous output, exactly like the XGBoost regressor above, as opposed to `MultiTaskNNRegressor` which instead predicts one output column per task/target — we replicate that merge by hand.

⚠️ A handful of mold2 descriptors that fail to compute are encoded as an enormous (~1e38) sentinel value instead of `null`. Fed straight into `StandardScaler`, that overflows the variance to `inf` and NaNs the whole network from then on (tree-based `pcm()` above is unaffected: a split just treats it as one huge value). We unmask those sentinels to `null` before scaling, and give every network its own checkpoint subfolder below:

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from papyrus_scripts.neuralnet import SingleTaskNNRegressor

merge_on = 'connectivity' if 'connectivity' in model_data.columns else 'InChIKey'

mol_descs = model_dataset.molecular_descriptors('mold2').to_dataframe()
SENTINEL_THRESHOLD = 1e30  # mold2's "could not compute" marker, not an exact float32 max
mol_descs = mol_descs.with_columns([
    pl.when(pl.col(c).abs() >= SENTINEL_THRESHOLD).then(None).otherwise(pl.col(c)).alias(c)
    for c in mol_descs.columns if c != merge_on
])
prot_descs = model_dataset.proteins().protein_descriptors('unirep')  # same descriptor as the 'unirep' pcm() run above

combined = (model_data
            .filter(pl.col('relation') == '=')
            .select([merge_on, 'target_id', 'pchembl_value_Mean'])
            .join(mol_descs, on=merge_on)
            .join(prot_descs, on='target_id')
            .drop_nulls())

feature_cols = [c for c in combined.columns if c not in (merge_on, 'target_id', 'pchembl_value_Mean')]
X = combined.select(feature_cols).to_numpy()
y = combined['pchembl_value_Mean'].to_numpy()
print(f'🧬 {X.shape[0]} datapoints × {X.shape[1]} combined mol+protein feature(s)')

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=1234)

reg = SingleTaskNNRegressor(out='./dnn_checkpoints/single_task_regressor', epochs=1500, lr=1e-3,
                             hidden_layers=[512, 128], dropout=0.15)
reg.set_architecture(n_dim=X_train.shape[1])
reg.set_validation(X_valid, y_valid)
reg.fit(X_train, y_train)

predictions = reg.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, predictions))
print(f'🧠 SingleTaskNNRegressor — R² {r2_score(y_valid, predictions):.3f}, RMSE {rmse:.3f}')

### 🧩 A different flavour of multi-task: `MultiTaskNNClassifier`

`pcm()` and the `SingleTaskNNRegressor` above are multi-task in the sense that *one* model is shared across every target, told apart by a protein descriptor merged onto each row. `MultiTaskNNClassifier` takes the other common approach: pivot `target_id` into one output *column* per task, so the network predicts every target's (now binary, thresholded at the usual `activity_threshold=6.5`) activity class at once from a single, target-independent input row — no protein descriptor needed at all, since which target is which is now encoded by output position rather than by an input feature. It reuses the exact same `mol_descs` (mold2) fetched for the regressor above.

Pivoted like this, the data is necessarily sparse — a compound is essentially never tested against *every* target in a panel, so most task columns are missing (`null`/`NaN`) for most compounds. Dropping every row with any missing task (as a naive `.drop_nulls()` would) throws away almost all of the data. Instead, `MultiTaskNNRegressor`/`MultiTaskNNClassifier` mask `NaN` targets out of the loss internally — a compound still contributes gradient for every task it *was* tested against, and simply contributes none for the ones it wasn't:

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

from papyrus_scripts.neuralnet import MultiTaskNNClassifier

mol_feature_cols = [c for c in mol_descs.columns if c != merge_on]

wide_data = (model_data
             .filter(pl.col('relation') == '=')
             .select([merge_on, 'target_id', 'pchembl_value_Mean'])
             .pivot(values='pchembl_value_Mean', index=merge_on, on='target_id', aggregate_function='mean')
             .join(mol_descs, on=merge_on)
             .drop_nulls(subset=mol_feature_cols))  # drop rows with bad/missing mold2 features only

task_cols = [c for c in wide_data.columns if c not in mol_feature_cols and c != merge_on]
X_multi = wide_data.select(mol_feature_cols).to_numpy()
raw_endpoint = wide_data.select(task_cols).to_numpy()
y_multi = np.where(np.isnan(raw_endpoint), np.nan, raw_endpoint > 6.5).astype('float32')  # NaN stays NaN
print(f'🧪 {wide_data.shape[0]} compound(s) × {len(task_cols)} task(s), '
      f'{100 * (~np.isnan(y_multi)).mean():.1f}% densely filled')

X_train, X_valid, y_train, y_valid = train_test_split(X_multi, y_multi, test_size=0.1, random_state=1234)

clf = MultiTaskNNClassifier(out='./dnn_checkpoints/multi_task_classifier', epochs=1500, lr=1e-3,
                             hidden_layers=[512, 128], dropout=0.15)
clf.set_architecture(n_dim=X_train.shape[1], n_task=len(task_cols))
clf.set_validation(X_valid, y_valid)
clf.fit(X_train, y_train)

predictions = clf.predict(X_valid)  # (n_compounds, n_tasks)
tested = ~np.isnan(y_valid)  # score only tested tasks
print(f'🧩 MultiTaskNNClassifier — accuracy {accuracy_score(y_valid[tested], predictions[tested]):.3f}')

---

🎉 That's ProDEC in a nutshell! For everything else `qsar()`/`pcm()` can do — `qsar()` itself, other splitting strategies, `yscramble`, repeated random seeds, and `MultiTaskNNRegressor` (the continuous-target counterpart of the classifier just above) — see [advanced_querying.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/advanced_querying.ipynb); for everyday filtering, see [simple_examples.ipynb](https://github.com/OlivierBeq/Papyrus-scripts/blob/master/notebook_examples/simple_examples.ipynb).